# 09 | RAG Retriever 检索器

前面我们已经把资料走到了这一步：

```text
资料 -> Loader -> Document -> Text Splitter -> chunks -> Embedding -> Vector Store
```

现在问题来了：用户提问时，谁负责从向量库里把相关 chunks 拿出来？

这就是 `Retriever`。

它在 RAG 里的位置是：

```text
用户问题 -> Retriever -> 相关 Documents -> Prompt -> LLM
```

Retriever 可以先理解成“检索入口”。它不负责生成回答，只负责根据用户问题返回相关文档。

这一步很关键。因为后面的 Prompt 不是凭空写的，它要靠 Retriever 拿到上下文。没有 Retriever，RAG 就像客服说“我查一下”，然后转身看空气。


# 一、Retriever 到底是什么

LangChain 里，Retriever 是一个统一接口：

```text
输入：一个问题字符串
输出：一组 Document
```

它可能来自向量库，也可能来自 BM25、搜索引擎、知识库 API。

这篇先讲最常见的一种：**由向量库转换出来的 Retriever**。

也就是：

```text
Vector Store -> as_retriever() -> Retriever
```


# 二、准备一份客服知识库

继续用一个客服知识库场景。用户可能会问退款、发票、会员续费、商品质量问题。

为了让这篇可以直接跑，先准备一份本地 TXT。


In [ ]:
from pathlib import Path

# 示例文件放在 rag/data 目录。
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

kb_path = data_dir / "support_policy_retriever.txt"

# 准备一份客服知识库文本。
# 真实项目里，这个文件通常来自客服后台、知识库系统或运营维护的文档。
kb_path.write_text(
    "售后知识库说明\n\n"
    "一、未发货订单退款\n"
    "如果订单还没有发货，用户可以在订单详情页申请退款。系统会自动拦截发货流程，退款通常会在 1 到 3 个工作日内原路退回。"
    "如果订单已经进入仓库打包状态，退款申请可能需要人工审核。\n\n"
    "二、已发货订单退款\n"
    "如果订单已经发货，用户需要先等待商品送达，再根据实际情况申请退货退款。"
    "退货时需要保持商品包装完整，并上传物流单号。仓库签收并验货通过后，退款会在 3 到 7 个工作日内退回。\n\n"
    "三、发票下载\n"
    "电子发票可以在订单详情页下载。如果用户找不到发票入口，可以引导用户进入：我的订单 -> 订单详情 -> 发票信息。"
    "如果发票抬头填写错误，需要联系人工客服重新开具。\n\n"
    "四、会员自动续费\n"
    "会员服务默认不会强制续费。用户如果开通了自动续费，可以在 App 的会员中心关闭。"
    "关闭自动续费后，已经购买的会员权益不会立即失效，会持续到当前会员周期结束。\n\n"
    "五、商品质量问题\n"
    "如果用户收到商品后发现破损、缺件或无法正常使用，需要在签收后 48 小时内提交售后申请。"
    "用户需要上传商品照片、外包装照片和问题说明。客服审核通过后，可以安排换货或退款。\n",
    encoding="utf-8",
)

print(kb_path)


# 三、加载、切分、写入向量库

Retriever 不是凭空工作的。它背后通常要有一个可检索的数据源。

这里先复用前面学过的流程：

```text
TXT -> TextLoader -> Document -> Text Splitter -> chunks -> Vector Store
```


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(str(kb_path), encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    # 这里把 chunk_size 控制得小一点，让每个业务小节尽量独立成块。
    chunk_size=120,
    # 保留少量重叠，避免切口处上下文断掉。
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
    # 优先按空行切分；这份知识库刚好每个业务小节之间都有空行。
    separators=["\n\n", "。", "，", " ", ""],
    add_start_index=True,
)

# Retriever 后面返回的就是这些 chunks。
chunks = splitter.split_documents(documents)

print("chunk 数量:", len(chunks))
for chunk in chunks:
    print("-" * 40)
    print(chunk.page_content)
    print(chunk.metadata)


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:latest",
    base_url="http://localhost:11434",
)

# 先把 chunks 写入内存向量库。
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)


# 四、先看不用 Retriever 的写法

前面我们一直是直接调用向量库：

```python
vectorstore.similarity_search(...)
```

这当然能用，但它把调用方和具体向量库绑得比较紧。


In [ ]:
results = vectorstore.similarity_search(
    "未发货订单能退款吗？",
    k=2,
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 40)


这个结果已经是对的。

但 RAG 后面要接 Prompt、Chain、Agent 时，通常不希望每个地方都知道你底层用的是哪个向量库。于是就有了 Retriever 这个统一入口。


# 五、把 Vector Store 转成 Retriever

向量库通常可以通过 `as_retriever()` 转成 Retriever。

当前 LangChain 推荐把 Retriever 当成 Runnable 使用，也就是调用：

```python
retriever.invoke("问题")
```

不要再想着它会直接生成答案。它只返回 `Document`。


In [ ]:
retriever = vectorstore.as_retriever(
    # search_kwargs 会传给底层向量搜索。
    # k=2 表示每次返回最相关的 2 个 chunk。
    search_kwargs={"k": 2}
)

retrieved_docs = retriever.invoke("未发货订单能退款吗？")

for doc in retrieved_docs:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 40)


你会发现：结果和 `similarity_search()` 很像。

区别不在结果，而在接口。

`vectorstore.similarity_search()` 是“直接问向量库”。

`retriever.invoke()` 是“通过统一检索器拿文档”。

后面把检索结果组装进 Prompt 时，用 Retriever 会更顺。


# 六、Retriever 返回的是 Document，不是答案

这是新手最容易误会的地方。

Retriever 的职责到这里就结束了：

```text
问题 -> 相关 Documents
```

它不会总结，不会润色，也不会替你回答。

如果你问：

```text
未发货订单能退款吗？
```

Retriever 只会把相关制度片段拿回来。真正把这些片段组织成一句自然语言回答，是下一步 Prompt + LLM 的工作。


In [ ]:
print(type(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0].page_content)


# 七、调整返回数量 k

`k` 控制 Retriever 返回多少个 chunks。

- `k` 太小：可能漏掉关键资料
- `k` 太大：可能把不相关内容也带进 Prompt

它不是越大越好。给模型塞太多资料，模型也会开始“眼神飘忽”。


In [ ]:
retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})
retriever_k3 = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "发票在哪里下载？"

k1_docs = retriever_k1.invoke(query)
k3_docs = retriever_k3.invoke(query)

print("k=1 返回 Document 数量:", len(k1_docs))
for i, doc in enumerate(k1_docs, start=1):
    print(f"# Document {i}")
    print(doc.page_content)
    print("-" * 40)

print("\nk=3 返回 Document 数量:", len(k3_docs))
for i, doc in enumerate(k3_docs, start=1):
    print(f"# Document {i}")
    print(doc.page_content)
    print("-" * 40)


实际项目里，`k` 要结合 chunk 大小、问题类型、Prompt 长度一起调。

## 小结下

1. Retriever 是 RAG 的检索入口。
2. 它输入问题，输出相关 `Document`。
3. 向量库可以通过 `as_retriever()` 转成 Retriever。
4. 当前用法是 `retriever.invoke("问题")`。
5. Retriever 不生成答案，只负责找资料。
6. `search_kwargs={"k": 2}` 可以控制返回数量。

到这里，Retriever 已经能返回相关 Documents。下一步就是把这些 Documents 组装进 Prompt：

用户问题 + Retriever 返回的上下文 -> Prompt -> LLM -> 回答

也就是说，Retriever 是 RAG 里的“查资料的人”。它查得准，模型才有东西可说。它查歪了，模型再会说也只能把歪资料说得很流畅。

这个场面我们都见过，甚至每天都见。